# Banking Fraud Intelligence
## September held-out evaluation

**P4 · September 2018 · Simulated card transactions**

Does the August comparison hold in September when both policies have the same number of review slots? Compare the amount baseline and the frozen logistic regression challenger at 50 and 100 reviews per day, then repeat the comparison for transactions at or below 220.

Use the same Python 3.12 kernel as the August modelling notebook and select **Run All**. [Notebook setup](../data/README.md#run-the-held-out-evaluation) · [Frozen protocol](../docs/methodology.md#frozen-model-and-review-protocol)

In [ ]:
from math import isclose
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

repo = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src" / "review_export.py").is_file()),
    None,
)
if repo is None:
    raise FileNotFoundError("Open this notebook from the Banking Fraud Intelligence checkout.")
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from src.fraud_model import POLICIES, RESTRICTED_COHORT, compare_review_queues, fit_ranker, score_transactions
from src.review_export import build_rank_export, save_rank_export
from src.transaction_features import build_features, split_features
from src.validate_transactions import load_transactions

## 1 · Reproduce the frozen fit

Build scoring-time features from the validated CSV. Fit the pipeline on April through July and score September. August events can supply earlier transaction history. September was previously examined in aggregate, and its model scores were reserved for this evaluation.

In [ ]:
features = build_features(load_transactions())
partitions = split_features(features)
training = partitions["train"]
holdout = partitions["test"]
model = fit_ranker(training)
scored = score_transactions(model, holdout)

assert training["TX_DATETIME"].max() < holdout["TX_DATETIME"].min()
assert scored["TX_DATETIME"].dt.normalize().nunique() == 30
pd.testing.assert_series_equal(scored["TRANSACTION_ID"], holdout["TRANSACTION_ID"])
assert np.isfinite(scored["model_score"]).all()
print(f"Fitted on {len(training):,} April–July transactions.")
print(f"Scored {len(scored):,} September transactions across 30 days.")

## 2 · Compare the September queues

For each cohort, both policies draw from the same eligible transactions. At or below 220, the queues are filled again from that cohort. Report fraud caught, legitimate reviews and fraud-labelled value from the selected rows. Recall uses the cohort's fraud total. Average precision evaluates the full ranking within each cohort.

In [ ]:
queues, average_precision = compare_review_queues(scored)
queue_display = queues.assign(
    Precision=queues["Precision"].map("{:.1%}".format),
    Recall=queues["Recall"].map("{:.1%}".format),
    **{"Fraud-labelled value caught": queues["Fraud-labelled value caught"].map("{:,.2f}".format)},
)
display(queue_display.set_index(["Cohort", "Daily limit", "Policy"]))
display(average_precision.assign(
    **{"Average precision": average_precision["Average precision"].map("{:.3f}".format)},
).set_index(["Cohort", "Policy"]))

comparison = queues.pivot(index=["Cohort", "Daily limit"], columns="Policy", values="Fraud caught")
difference = (comparison["Logistic regression"] - comparison["Amount baseline"]).unstack("Cohort")
figure, axis = plt.subplots(figsize=(8.2, 3.6), layout="constrained")
difference.plot.bar(ax=axis, color=["#137A73", "#5B6B89"], rot=0)
axis.axhline(0, color="#334155", linewidth=0.8)
axis.set(title="September holdout: change in fraud caught", ylabel="Model minus amount baseline")
axis.set_xlabel("Daily review limit")
axis.legend(frameon=False, title=None)
plt.show()

## 3 · Reconcile the ranks and save the local export

Filter the export by each of its four rank columns at each capacity. Verify that the selected rows reproduce the September report, that each day fills the same capacity under both policies, and that the restricted ranks belong only to eligible rows.

The CSV is written to the Git-ignored `data/processed/` directory. The notebook displays aggregate results only. Fraud-labelled value represents the transaction amounts selected for review.

In [ ]:
export = build_rank_export(scored)
assert export["TRANSACTION_ID"].is_unique
assert export["TRANSACTION_ID"].tolist() == scored["TRANSACTION_ID"].tolist()
rank_columns = {
    (cohort, policy): f"{score_column.lower()}_rank_{suffix}"
    for cohort, suffix in (("All transactions", "all"), (RESTRICTED_COHORT, "le_220"))
    for policy, score_column in POLICIES.items()
}
restricted_eligible = export["TX_AMOUNT"] <= 220
for (cohort, _), column in rank_columns.items():
    eligible = restricted_eligible if cohort == RESTRICTED_COHORT else pd.Series(True, index=export.index)
    pd.testing.assert_series_equal(export[column].notna(), eligible, check_names=False)
    ranked = export.loc[eligible, ["TX_DATETIME", column]]
    day = ranked["TX_DATETIME"].dt.normalize()
    ranks_by_day = ranked.groupby(day)[column]
    assert ranks_by_day.min().eq(1).all()
    assert ranks_by_day.nunique().eq(ranks_by_day.size()).all()
    assert ranks_by_day.max().eq(ranks_by_day.size()).all()

for outcome in queues.to_dict("records"):
    limit = outcome["Daily limit"]
    rank = export[rank_columns[(outcome["Cohort"], outcome["Policy"])]]
    selected = export.loc[rank.le(limit)]
    caught = selected.loc[selected["TX_FRAUD"].eq(1)]
    assert selected.groupby(selected["TX_DATETIME"].dt.normalize()).size().eq(limit).all()
    assert len(selected) == outcome["Reviews"] == 30 * limit
    assert len(caught) == outcome["Fraud caught"]
    assert len(selected) - len(caught) == outcome["Legitimate reviews"]
    assert isclose(float(caught["TX_AMOUNT"].sum()), outcome["Fraud-labelled value caught"], rel_tol=0, abs_tol=1e-8)

output_path = repo / "data" / "processed" / "september_review_ranks.csv"
save_rank_export(export, output_path)
print(f"Verified all eight queue comparisons and saved {len(export):,} rows to {output_path.relative_to(repo)}.")

## Next checkpoint

Use the reconciled results to decide which Power BI views and interactions to build. [Project handoff](../docs/SESSION_HANDOFF_CURRENT.md)